# Modelagem da camada Gold

######Tabela Gold: gold_trip_operations

######Granularidade: uma linha por viagem.

######Objetivo: consolidar dados operacionais, financeiros e cadastrais das viagens para análises de desempenho.

##Ler tabelas da camada Silver

In [0]:
from pyspark.sql import functions as F

In [0]:
df_trips_silver = spark.table(
    "workspace.silver_logistics.trips"
)

df_loads_silver = spark.table(
    "workspace.silver_logistics.loads"
)

df_routes_silver = spark.table(
    "workspace.silver_logistics.routes"
)

df_drivers_silver = spark.table(
    "workspace.silver_logistics.drivers"
)

df_trucks_silver = spark.table(
    "workspace.silver_logistics.trucks"
)

##Validar volumes das tabelas de origem

In [0]:
print(f"Trips: {df_trips_silver.count()}")
print(f"Loads: {df_loads_silver.count()}")
print(f"Routes: {df_routes_silver.count()}")
print(f"Drivers: {df_drivers_silver.count()}")
print(f"Trucks: {df_trucks_silver.count()}")

##Verificar colunas disponíveis nas tabelas Silver

In [0]:
print("TRIPS")
print(df_trips_silver.columns)

print("\nLOADS")
print(df_loads_silver.columns)

print("\nROUTES")
print(df_routes_silver.columns)

print("\nDRIVERS")
print(df_drivers_silver.columns)

print("\nTRUCKS")
print(df_trucks_silver.columns)

##Preparar dados de viagens para a Gold

In [0]:
df_trips_gold_base = (
    df_trips_silver.select(
        "trip_id",
        "load_id",
        "driver_id",
        "truck_id",
        "trailer_id",
        "dispatch_date",
        "actual_distance_miles",
        "actual_duration_hours",
        "fuel_gallons_used",
        "average_mpg",
        "idle_time_hours",
        "trip_status",
        "trip_year",
        "trip_month",
        "trip_quarter",
        "trip_year_month",
        "calculated_mpg",
        "average_speed_mph",
        "idle_time_percentage",
        "mpg_difference",
        "has_driver",
        "has_truck",
        "has_trailer",
        "has_complete_resource_assignment",
        "idle_time_inconsistent"
    )
)

##Preparar dados de cargas

In [0]:
df_loads_gold_base = (
    df_loads_silver.select(
        "load_id",
        "customer_id",
        "route_id",
        "load_date",
        "load_type",
        "weight_lbs",
        "pieces",
        "revenue",
        "fuel_surcharge",
        "accessorial_charges",
        "load_status",
        "booking_type",
        "total_revenue",
        "revenue_per_piece",
        "revenue_per_lb",
        "has_complete_relationship"
    )
)

##Preparar dados de rotas

In [0]:
df_routes_gold_base = (
    df_routes_silver.select(
        "route_id",
        "origin_location",
        "destination_location",
        "route_description",
        "typical_distance_miles",
        "base_rate_per_mile",
        "fuel_surcharge_rate",
        "typical_transit_days",
        "estimated_rate_per_mile",
        "estimated_route_cost",
        "average_miles_per_transit_day",
        "has_complete_route"
    )
)

##Preparar dados de motoristas

In [0]:
df_drivers_gold_base = (
    df_drivers_silver.select(
        "driver_id",
        F.concat_ws(
            " ",
            F.col("first_name"),
            F.col("last_name")
        ).alias("driver_name"),
        F.col("home_terminal").alias("driver_home_terminal"),
        F.col("employment_status").alias("driver_employment_status"),
        "cdl_class",
        "years_experience",
        "driver_age",
        "tenure_years",
        "is_active_driver",
        "has_valid_license"
    )
)

##Preparar dados de caminhões

In [0]:
df_trucks_gold_base = (
    df_trucks_silver.select(
        "truck_id",
        "unit_number",
        F.col("make").alias("truck_make"),
        "model_year",
        F.col("fuel_type").alias("truck_fuel_type"),
        F.col("status").alias("truck_status"),
        F.col("home_terminal").alias("truck_home_terminal"),
        "truck_age_years",
        "years_in_fleet",
        "is_active_truck",
        "has_complete_truck_record"
    )
)

##Consolidar viagens, cargas e rotas

In [0]:
df_gold_trip_operations_base = (
    df_trips_gold_base.alias("trip")
    .join(
        df_loads_gold_base.alias("load"),
        on="load_id",
        how="left"
    )
    .join(
        df_routes_gold_base.alias("route"),
        on="route_id",
        how="left"
    )
)

##Adicionar dados de motoristas e caminhões

In [0]:
df_gold_trip_operations_enriquecida = (
    df_gold_trip_operations_base.alias("operation")
    .join(
        df_drivers_gold_base.alias("driver"),
        on="driver_id",
        how="left"
    )
    .join(
        df_trucks_gold_base.alias("truck"),
        on="truck_id",
        how="left"
    )
)

##Criar métricas de negócio da operação

In [0]:
df_gold_trip_operations = (
    df_gold_trip_operations_enriquecida
    .withColumn(
        "distance_variance_miles",
        F.round(
            F.col("actual_distance_miles")
            - F.col("typical_distance_miles"),
            2
        )
    )
    .withColumn(
        "distance_variance_percentage",
        F.when(
            F.col("typical_distance_miles") > 0,
            F.round(
                (
                    F.col("actual_distance_miles")
                    - F.col("typical_distance_miles")
                )
                / F.col("typical_distance_miles")
                * 100,
                2
            )
        )
    )
    .withColumn(
        "estimated_operational_margin",
        F.round(
            F.col("total_revenue")
            - F.col("estimated_route_cost"),
            2
        )
    )
    .withColumn(
        "estimated_margin_percentage",
        F.when(
            F.col("total_revenue") > 0,
            F.round(
                (
                    F.col("total_revenue")
                    - F.col("estimated_route_cost")
                )
                / F.col("total_revenue")
                * 100,
                2
            )
        )
    )
    .withColumn(
        "revenue_per_actual_mile",
        F.when(
            F.col("actual_distance_miles") > 0,
            F.round(
                F.col("total_revenue")
                / F.col("actual_distance_miles"),
                2
            )
        )
    )
    .withColumn(
        "has_complete_gold_relationship",
        F.col("load_id").isNotNull()
        & F.col("route_id").isNotNull()
    )
)

##Validar preservação da granularidade

In [0]:
print(
    "Total de viagens na Silver: "
    f"{df_trips_silver.count()}"
)

print(
    "Total de registros na Gold: "
    f"{df_gold_trip_operations.count()}"
)

print(
    "Diferença de registros: "
    f"{df_gold_trip_operations.count() - df_trips_silver.count()}"
)

##Validar chave e relacionamentos da tabela Gold

In [0]:
display(
    df_gold_trip_operations.select(
        F.count("*").alias("total_linhas"),

        F.countDistinct("trip_id").alias(
            "trip_ids_unicos"
        ),

        F.sum(
            F.when(
                F.col("trip_id").isNull(),
                1
            ).otherwise(0)
        ).alias("trip_ids_nulos"),

        F.sum(
            F.when(
                F.col("load_id").isNull(),
                1
            ).otherwise(0)
        ).alias("loads_ausentes"),

        F.sum(
            F.when(
                F.col("route_id").isNull(),
                1
            ).otherwise(0)
        ).alias("routes_ausentes"),

        F.sum(
            F.when(
                ~F.col("has_complete_gold_relationship"),
                1
            ).otherwise(0)
        ).alias("relacionamentos_gold_incompletos")
    )
)

##Visualizar resultado da gold_trip_operations

In [0]:
display(
    df_gold_trip_operations.select(
        "trip_id",
        "dispatch_date",
        "trip_year_month",
        "customer_id",
        "route_id",
        "route_description",
        "driver_id",
        "driver_name",
        "truck_id",
        "unit_number",
        "trip_status",
        "load_type",
        "actual_distance_miles",
        "typical_distance_miles",
        "distance_variance_miles",
        "distance_variance_percentage",
        "actual_duration_hours",
        "average_speed_mph",
        "fuel_gallons_used",
        "calculated_mpg",
        "idle_time_percentage",
        "total_revenue",
        "estimated_route_cost",
        "estimated_operational_margin",
        "estimated_margin_percentage",
        "revenue_per_actual_mile",
        "has_complete_resource_assignment",
        "has_complete_gold_relationship"
    ).limit(20)
)

##Validar métricas calculadas da Gold

In [0]:
display(
    df_gold_trip_operations.select(
        F.sum(
            F.when(
                F.col("distance_variance_percentage").isNull(),
                1
            ).otherwise(0)
        ).alias("variacoes_distancia_nulas"),

        F.sum(
            F.when(
                F.col("revenue_per_actual_mile") <= 0,
                1
            ).otherwise(0)
        ).alias("receitas_por_milha_invalidas"),

        F.sum(
            F.when(
                F.col("estimated_margin_percentage").isNull(),
                1
            ).otherwise(0)
        ).alias("margens_percentuais_nulas")
    )
)

##Analisar margem operacional estimada

In [0]:
display(
    df_gold_trip_operations.select(
        F.count("*").alias("total_viagens"),

        F.sum(
            F.when(
                F.col("estimated_operational_margin") > 0,
                1
            ).otherwise(0)
        ).alias("viagens_com_margem_positiva"),

        F.sum(
            F.when(
                F.col("estimated_operational_margin") < 0,
                1
            ).otherwise(0)
        ).alias("viagens_com_margem_negativa"),

        F.round(
            F.avg("estimated_operational_margin"),
            2
        ).alias("margem_media_estimada"),

        F.round(
            F.avg("estimated_margin_percentage"),
            2
        ).alias("margem_percentual_media")
    )
)

##Analisar desvio de distância

In [0]:
display(
    df_gold_trip_operations.select(
        F.round(
            F.avg("distance_variance_miles"),
            2
        ).alias("desvio_medio_milhas"),

        F.round(
            F.avg("distance_variance_percentage"),
            2
        ).alias("desvio_medio_percentual"),

        F.round(
            F.max("distance_variance_percentage"),
            2
        ).alias("maior_desvio_percentual"),

        F.round(
            F.min("distance_variance_percentage"),
            2
        ).alias("menor_desvio_percentual")
    )
)

##Validar recursos associados às viagens

In [0]:
display(
    df_gold_trip_operations.select(
        F.sum(
            F.when(
                F.col("has_driver"),
                1
            ).otherwise(0)
        ).alias("viagens_com_motorista"),

        F.sum(
            F.when(
                ~F.col("has_driver"),
                1
            ).otherwise(0)
        ).alias("viagens_sem_motorista"),

        F.sum(
            F.when(
                F.col("has_truck"),
                1
            ).otherwise(0)
        ).alias("viagens_com_caminhao"),

        F.sum(
            F.when(
                ~F.col("has_truck"),
                1
            ).otherwise(0)
        ).alias("viagens_sem_caminhao"),

        F.sum(
            F.when(
                F.col("has_complete_resource_assignment"),
                1
            ).otherwise(0)
        ).alias("viagens_com_recursos_completos")
    )
)

##Gravar gold_trip_operations

In [0]:
(
    df_gold_trip_operations.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.gold_logistics.gold_trip_operations"
    )
)

##Validar gravação da gold_trip_operations

In [0]:
df_gold_trip_operations_gravada = spark.table(
    "workspace.gold_logistics.gold_trip_operations"
)

print(
    "Total de registros gravados na Gold: "
    f"{df_gold_trip_operations_gravada.count()}"
)

df_gold_trip_operations_gravada.printSchema()

##Validar estrutura persistida

In [0]:
display(
    df_gold_trip_operations_gravada.select(
        F.count("*").alias("total_linhas"),

        F.countDistinct("trip_id").alias(
            "trip_ids_unicos"
        ),

        F.sum(
            F.when(
                F.col("trip_id").isNull(),
                1
            ).otherwise(0)
        ).alias("trip_ids_nulos"),

        F.sum(
            F.when(
                ~F.col("has_complete_gold_relationship"),
                1
            ).otherwise(0)
        ).alias("relacionamentos_incompletos")
    )
)

######Tabela Gold: gold_delivery_performance

######Granularidade: uma linha por viagem.

######Objetivo: consolidar os eventos de Pickup e Delivery para medir pontualidade, atrasos, antecipações e detenção.

##Ler tabela delivery_events da Silver

In [0]:
df_delivery_events_silver = spark.table(
    "workspace.silver_logistics.delivery_events"
)

##Separar eventos de Pickup

In [0]:
df_pickup = (
    df_delivery_events_silver
    .filter(
        F.col("event_type") == "Pickup"
    )
    .select(
        "trip_id",
        F.col("scheduled_datetime").alias("pickup_scheduled_datetime"),
        F.col("actual_datetime").alias("pickup_actual_datetime"),
        F.col("delay_minutes").alias("pickup_delay_minutes"),
        F.col("early_minutes").alias("pickup_early_minutes"),
        F.col("detention_minutes").alias("pickup_detention_minutes"),
        F.col("on_time_flag").alias("pickup_on_time_flag_source"),
        F.col("is_on_time_calculated").alias("pickup_on_time_calculated"),
        F.col("on_time_flag_consistent").alias("pickup_flag_consistent"),
        F.col("facility_id").alias("pickup_facility_id"),
        F.col("location").alias("pickup_location")
    )
)

##Separar eventos de Delivery

In [0]:
df_delivery = (
    df_delivery_events_silver
    .filter(
        F.col("event_type") == "Delivery"
    )
    .select(
        "trip_id",
        F.col("scheduled_datetime").alias("delivery_scheduled_datetime"),
        F.col("actual_datetime").alias("delivery_actual_datetime"),
        F.col("delay_minutes").alias("delivery_delay_minutes"),
        F.col("early_minutes").alias("delivery_early_minutes"),
        F.col("detention_minutes").alias("delivery_detention_minutes"),
        F.col("on_time_flag").alias("delivery_on_time_flag_source"),
        F.col("is_on_time_calculated").alias("delivery_on_time_calculated"),
        F.col("on_time_flag_consistent").alias("delivery_flag_consistent"),
        F.col("facility_id").alias("delivery_facility_id"),
        F.col("location").alias("delivery_location")
    )
)

##Consolidar Pickup e Delivery por viagem

In [0]:
df_gold_delivery_performance = (
    df_pickup
    .join(
        df_delivery,
        on="trip_id",
        how="inner"
    )
)

##Criar indicadores de desempenho da entrega

In [0]:
df_gold_delivery_performance = (
    df_gold_delivery_performance
    .withColumn(
        "total_detention_minutes",
        F.col("pickup_detention_minutes")
        + F.col("delivery_detention_minutes")
    )
    .withColumn(
        "has_pickup_delay",
        F.col("pickup_delay_minutes") > 0
    )
    .withColumn(
        "has_delivery_delay",
        F.col("delivery_delay_minutes") > 0
    )
    .withColumn(
        "has_any_delay",
        (F.col("pickup_delay_minutes") > 0)
        | (F.col("delivery_delay_minutes") > 0)
    )
    .withColumn(
        "both_events_on_time",
        F.col("pickup_on_time_calculated")
        & F.col("delivery_on_time_calculated")
    )
    .withColumn(
        "has_source_flag_inconsistency",
        ~F.col("pickup_flag_consistent")
        | ~F.col("delivery_flag_consistent")
    )
    .withColumn(
        "delivery_cycle_hours",
        F.round(
            (
                F.unix_timestamp("delivery_actual_datetime")
                - F.unix_timestamp("pickup_actual_datetime")
            ) / 3600,
            2
        )
    )
)

##Validar granularidade da tabela Gold

In [0]:
display(
    df_gold_delivery_performance.select(
        F.count("*").alias("total_linhas"),
        F.countDistinct("trip_id").alias("trip_ids_unicos"),
        F.sum(
            F.when(
                F.col("trip_id").isNull(),
                1
            ).otherwise(0)
        ).alias("trip_ids_nulos")
    )
)

##Validar métricas operacionais

In [0]:
display(
    df_gold_delivery_performance.select(
        F.sum(
            F.when(
                F.col("has_pickup_delay"),
                1
            ).otherwise(0)
        ).alias("viagens_com_atraso_pickup"),

        F.sum(
            F.when(
                F.col("has_delivery_delay"),
                1
            ).otherwise(0)
        ).alias("viagens_com_atraso_delivery"),

        F.sum(
            F.when(
                F.col("has_any_delay"),
                1
            ).otherwise(0)
        ).alias("viagens_com_algum_atraso"),

        F.sum(
            F.when(
                F.col("both_events_on_time"),
                1
            ).otherwise(0)
        ).alias("viagens_totalmente_no_prazo"),

        F.sum(
            F.when(
                F.col("has_source_flag_inconsistency"),
                1
            ).otherwise(0)
        ).alias("viagens_com_divergencia_flag")
    )
)

##Analisar médias de atraso e detenção

In [0]:
display(
    df_gold_delivery_performance.select(
        F.round(
            F.avg("pickup_delay_minutes"),
            2
        ).alias("media_atraso_pickup"),

        F.round(
            F.avg("delivery_delay_minutes"),
            2
        ).alias("media_atraso_delivery"),

        F.round(
            F.avg("total_detention_minutes"),
            2
        ).alias("media_detencao_total"),

        F.round(
            F.avg("delivery_cycle_hours"),
            2
        ).alias("media_ciclo_entrega_horas")
    )
)

##Validar duração do ciclo de entrega

In [0]:
display(
    df_gold_delivery_performance.select(
        F.sum(
            F.when(
                F.col("delivery_cycle_hours") < 0,
                1
            ).otherwise(0)
        ).alias("ciclos_entrega_negativos"),

        F.sum(
            F.when(
                F.col("delivery_cycle_hours").isNull(),
                1
            ).otherwise(0)
        ).alias("ciclos_entrega_nulos")
    )
)

##Visualizar resultado final

In [0]:
display(
    df_gold_delivery_performance.select(
        "trip_id",
        "pickup_location",
        "delivery_location",
        "pickup_scheduled_datetime",
        "pickup_actual_datetime",
        "pickup_delay_minutes",
        "pickup_early_minutes",
        "delivery_scheduled_datetime",
        "delivery_actual_datetime",
        "delivery_delay_minutes",
        "delivery_early_minutes",
        "pickup_detention_minutes",
        "delivery_detention_minutes",
        "total_detention_minutes",
        "delivery_cycle_hours",
        "has_pickup_delay",
        "has_delivery_delay",
        "has_any_delay",
        "both_events_on_time",
        "has_source_flag_inconsistency"
    ).limit(20)
)

##Gravar gold_delivery_performance

In [0]:
(
    df_gold_delivery_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.gold_logistics.gold_delivery_performance"
    )
)

##Validar gravação

In [0]:
df_gold_delivery_performance_gravada = spark.table(
    "workspace.gold_logistics.gold_delivery_performance"
)

print(
    "Total de registros gravados na Gold: "
    f"{df_gold_delivery_performance_gravada.count()}"
)

df_gold_delivery_performance_gravada.printSchema()

##Validar estrutura persistida

In [0]:
display(
    df_gold_delivery_performance_gravada.select(
        F.count("*").alias("total_linhas"),

        F.countDistinct("trip_id").alias("trip_ids_unicos"),

        F.sum(
            F.when(
                F.col("trip_id").isNull(),
                1
            ).otherwise(0)
        ).alias("trip_ids_nulos")
    )
)

######Tabela Gold: gold_fuel_performance

######Granularidade: uma linha por viagem.

######Objetivo: consolidar consumo e custo de combustível por viagem, além de indicadores de completude e consistência.

##Ler fuel_purchases da Silver

In [0]:
df_fuel_purchases_silver = spark.table(
    "workspace.silver_logistics.fuel_purchases"
)

##Agregar compras de combustível por viagem

In [0]:
df_fuel_por_trip = (
    df_fuel_purchases_silver
    .groupBy("trip_id")
    .agg(
        F.count("*").alias("fuel_purchase_count"),

        F.round(
            F.sum("gallons"),
            2
        ).alias("total_gallons_purchased"),

        F.round(
            F.sum("total_cost"),
            2
        ).alias("total_fuel_cost"),

        F.round(
            F.avg("price_per_gallon"),
            3
        ).alias("average_price_per_gallon"),

        F.round(
            F.avg("gallons"),
            2
        ).alias("average_gallons_per_purchase"),

        F.sum(
            F.when(
                ~F.col("has_truck"),
                1
            ).otherwise(0)
        ).alias("purchases_without_truck"),

        F.sum(
            F.when(
                ~F.col("has_driver"),
                1
            ).otherwise(0)
        ).alias("purchases_without_driver"),

        F.sum(
            F.when(
                ~F.col("has_consistent_total_cost"),
                1
            ).otherwise(0)
        ).alias("inconsistent_cost_purchases"),

        F.sum(
            F.when(
                F.col("truck_matches_trip") == False,
                1
            ).otherwise(0)
        ).alias("truck_mismatch_purchases"),

        F.sum(
            F.when(
                F.col("driver_matches_trip") == False,
                1
            ).otherwise(0)
        ).alias("driver_mismatch_purchases")
    )
)

##Adicionar dados operacionais da viagem

In [0]:
df_gold_fuel_performance = (
    df_trips_silver.select(
        "trip_id",
        "dispatch_date",
        "trip_year",
        "trip_month",
        "trip_year_month",
        "driver_id",
        "truck_id",
        "actual_distance_miles",
        "fuel_gallons_used",
        "calculated_mpg",
        "average_mpg"
    )
    .join(
        df_fuel_por_trip,
        on="trip_id",
        how="left"
    )
)

##Criar métricas de combustível

In [0]:
df_gold_fuel_performance = (
    df_gold_fuel_performance
    .withColumn(
        "fuel_cost_per_mile",
        F.when(
            F.col("actual_distance_miles") > 0,
            F.round(
                F.col("total_fuel_cost")
                / F.col("actual_distance_miles"),
                2
            )
        )
    )
    .withColumn(
        "purchased_vs_used_gallons_difference",
        F.round(
            F.col("total_gallons_purchased")
            - F.col("fuel_gallons_used"),
            2
        )
    )
    .withColumn(
        "has_fuel_purchase",
        F.col("fuel_purchase_count").isNotNull()
    )
    .withColumn(
        "has_complete_fuel_data",
        F.col("fuel_purchase_count").isNotNull()
        & F.col("total_gallons_purchased").isNotNull()
        & F.col("total_fuel_cost").isNotNull()
    )
)

In [0]:
df_gold_fuel_performance = (
    df_gold_fuel_performance
    .fillna(
        {
            "fuel_purchase_count": 0,
            "purchases_without_truck": 0,
            "purchases_without_driver": 0,
            "inconsistent_cost_purchases": 0,
            "truck_mismatch_purchases": 0,
            "driver_mismatch_purchases": 0
        }
    )
)

##Validar granularidade

In [0]:
display(
    df_gold_fuel_performance.select(
        F.count("*").alias("total_linhas"),

        F.countDistinct("trip_id").alias(
            "trip_ids_unicos"
        ),

        F.sum(
            F.when(
                F.col("trip_id").isNull(),
                1
            ).otherwise(0)
        ).alias("trip_ids_nulos")
    )
)

##Validar cobertura de compras de combustível

In [0]:
display(
    df_gold_fuel_performance.select(
        F.sum(
            F.when(
                F.col("has_fuel_purchase"),
                1
            ).otherwise(0)
        ).alias("viagens_com_compra_combustivel"),

        F.sum(
            F.when(
                ~F.col("has_fuel_purchase"),
                1
            ).otherwise(0)
        ).alias("viagens_sem_compra_combustivel"),

        F.sum(
            F.when(
                F.col("has_complete_fuel_data"),
                1
            ).otherwise(0)
        ).alias("viagens_com_dados_combustivel_completos")
    )
)

##Validar métricas financeiras

In [0]:
display(
    df_gold_fuel_performance.select(
        F.sum(
            F.when(
                (F.col("total_fuel_cost") <= 0)
                & F.col("has_fuel_purchase"),
                1
            ).otherwise(0)
        ).alias("custos_combustivel_invalidos"),

        F.sum(
            F.when(
                (F.col("total_gallons_purchased") <= 0)
                & F.col("has_fuel_purchase"),
                1
            ).otherwise(0)
        ).alias("galoes_comprados_invalidos"),

        F.sum(
            F.when(
                F.col("fuel_cost_per_mile") < 0,
                1
            ).otherwise(0)
        ).alias("custos_por_milha_negativos")
    )
)

##Analisar métricas gerais de combustível

In [0]:
display(
    df_gold_fuel_performance.select(
        F.sum("fuel_purchase_count").alias(
            "total_compras_combustivel"
        ),

        F.round(
            F.sum("total_gallons_purchased"),
            2
        ).alias("total_galoes_comprados"),

        F.round(
            F.sum("total_fuel_cost"),
            2
        ).alias("custo_total_combustivel"),

        F.round(
            F.avg("average_price_per_gallon"),
            3
        ).alias("preco_medio_por_galao"),

        F.round(
            F.avg("fuel_cost_per_mile"),
            2
        ).alias("custo_medio_combustivel_por_milha"),

        F.round(
            F.avg("calculated_mpg"),
            2
        ).alias("mpg_medio")
    )
)

##Visualizar resultado final

In [0]:
display(
    df_gold_fuel_performance.select(
        "trip_id",
        "dispatch_date",
        "trip_year_month",
        "driver_id",
        "truck_id",
        "actual_distance_miles",
        "fuel_gallons_used",
        "calculated_mpg",
        "fuel_purchase_count",
        "total_gallons_purchased",
        "total_fuel_cost",
        "average_price_per_gallon",
        "fuel_cost_per_mile",
        "purchased_vs_used_gallons_difference",
        "has_fuel_purchase",
        "has_complete_fuel_data",
        "purchases_without_truck",
        "purchases_without_driver",
        "inconsistent_cost_purchases",
        "truck_mismatch_purchases",
        "driver_mismatch_purchases"
    ).limit(20)
)

##Gravar gold_fuel_performance

In [0]:
(
    df_gold_fuel_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.gold_logistics.gold_fuel_performance"
    )
)

##Validar gravação

In [0]:
df_gold_fuel_performance_gravada = spark.table(
    "workspace.gold_logistics.gold_fuel_performance"
)

print(
    "Total de registros gravados na Gold: "
    f"{df_gold_fuel_performance_gravada.count()}"
)

df_gold_fuel_performance_gravada.printSchema()

##Validar estrutura persistida

In [0]:
display(
    df_gold_fuel_performance_gravada.select(
        F.count("*").alias("total_linhas"),

        F.countDistinct("trip_id").alias(
            "trip_ids_unicos"
        ),

        F.sum(
            F.when(
                F.col("trip_id").isNull(),
                1
            ).otherwise(0)
        ).alias("trip_ids_nulos")
    )
)